# Package-based walkthrough — the in-memory / UI flow

This notebook walks through the **recommended onboarding flow** for the
Explainability Toolbox when it is used as an installed Python package (for
example behind a web UI, or inside a Databricks / Colab notebook).

The flow is always the same four moves:

1. **Inspect** the uploaded model (and, ideally, a labelled data table). The
   toolbox reads whatever metadata it can and *auto-fills* the config.
2. **Ask** the user only for what could not be inferred. `inspect_model`
   returns that list as structured field specs — one per UI form control.
3. **Resolve** the inferred values + the user's answers into a single config.
4. **Explain** — run the analysis and get results back in memory.

> **For the UI developer:** every item in `profile.missing` is a form field.
> Its `type` tells you which control to render (checkbox, dropdown, multiselect,
> number, text), `options`/`default` pre-populate it, and `prompt`/`reason`
> are the label and helper text. Nothing here blocks on user input, so the same
> code backs a web form, a notebook, or a CLI.

## 0. Install

Install the package and the backend your model needs.

In [ ]:
# %pip install "git+https://github.com/SavvinaFil/AI-Explainability@v0.4.0"
# For a PyTorch model also:  %pip install "ai-explainability[torch]"
# For a TensorFlow model:    %pip install "ai-explainability[tensorflow]"
import ai_explainability as aie
print("toolbox version:", aie.__version__)

## 1. Load your model and data (in memory)

Here we train a small model as a stand-in for the user's uploaded model. In
the real app, `model` is whatever the user uploaded (unpickled / loaded) and
`df` is the table they provided.

**Why the data header matters:** if `df` has named columns, the toolbox reads
the feature names straight from the header — no separate naming step, and it
can cross-check the column count against the model's expected input width.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression

FEATURES = ["Wind_Speed", "Temp", "Humidity", "Pressure"]
X, y = make_regression(n_samples=300, n_features=4, n_targets=3, noise=0.1, random_state=0)
df = pd.DataFrame(X, columns=FEATURES)          # <- labelled header
model = RandomForestRegressor(n_estimators=50, random_state=0).fit(df, y)

## 2. Inspect — auto-fill what we can

`inspect_model(model, data)` returns a `ModelProfile` with three parts:

- `profile.inferred` — config keys we're confident about (pre-filled fields).
- `profile.missing`  — the fields the UI must still ask for.
- `profile.warnings` — soft problems, e.g. a data/model dimension mismatch.

`profile.describe()` prints a human summary; `profile.to_dict()` is the
JSON contract a front-end would consume.

In [ ]:
profile = aie.inspect_model(model, df)
print(profile.describe())

In [ ]:
# The exact JSON a UI would render:
import json
print(json.dumps(profile.to_dict(), indent=2, default=str))

## 3. Ask the user — render `profile.missing` as a form

Each entry maps to one control:

| `type`     | UI control                    |
|------------|-------------------------------|
| `bool`     | checkbox / toggle             |
| `int`      | number input                  |
| `str`      | text input                    |
| `enum`     | single-select dropdown        |
| `list`     | multi-select / ordered list   |
| `columns`  | column picker over the data   |

Here we answer them programmatically (the UI would collect these from the
form). Note `feature_names` with `type="columns"` is exactly the "select which
columns to explain" interaction — the user can narrow the feature set.

In [ ]:
answers = {
    # multi-output model -> which targets to explain (defaults to all)
    "target_index": [0, 1, 2],
    "output_labels": {"0": "Power", "1": "Load", "2": "Frequency"},
    # operational toggles (default False -> fully in-memory)
    "save_excel": False,
    "generate_notebook": False,
}
# feature_names was already inferred from the header, so we don't need to ask.
print("inferred feature_names:", profile.inferred["feature_names"])

## 4. Resolve — inferred values + answers -> one config

In [ ]:
config = aie.resolve_config(profile, answers)
print(json.dumps(config, indent=2, default=str))

## 5. Explain — run it

`config` is designed to splat straight into `aie.explain()`. The `model` and
`data` objects are passed alongside it (they aren't part of the serialisable
config).

In [ ]:
result = aie.explain(model=model, data=df.iloc[:50], **config)
result.to_dataframe().head()

## 6. Use the results

- `result.shap_values` — `dict[target -> ndarray]`.
- `result.predictions` — `dict[target -> ndarray]`.
- `result.to_dataframe()` — one flat table, great for `display()`.
- `result.save_excel(path)` / `result.save_notebook(path)` — opt-in disk output.

Nothing was written to disk in this run.

In [ ]:
for t, arr in result.shap_values.items():
    print(f"target {t}: SHAP {arr.shape}")
# result.save_excel("audit.xlsx")   # uncomment to write an Excel audit

## Summary for the UI

1. Upload model (+ optional labelled data) -> call `inspect_model`.
2. Show `profile.inferred` as pre-filled, read-only-ish fields.
3. Render `profile.missing` as a form (table above maps type -> control).
4. Surface `profile.warnings` as inline banners.
5. On submit -> `resolve_config(profile, answers)` -> `aie.explain(model, data, **config)`.
6. Render `result.to_dataframe()`; offer "Download Excel" -> `result.save_excel(...)`.